# C1 02 응답 수집 (T4)
프롬프트 200개 × 10회를 모델 하나씩 돌려서 Drive에 저장. 패키지 추출이랑 판정은 다음 단계

- 런타임 T4 GPU로 바꾸고 수집 셀 하나만 실행
- 모델은 셀 맨 위 MODEL 값만 바꾸면 됨
  - Qwen/Qwen2.5-Coder-7B-Instruct-AWQ
  - TheBloke/deepseek-coder-6.7B-instruct-AWQ
  - LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct-AWQ
- 끊기면 다시 실행. 200개 끝날 때마다 저장돼서 이어서 돎
- 모델 하나에 2.5~4.5시간 정도. 탭은 열어 둘 것

In [ ]:
MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct-AWQ'
MAX_HOURS = 10
CODE = '/content/drive/MyDrive/C1/code'
OUT = '/content/drive/MyDrive/C1/results/02_collect'

!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
import os; os.makedirs(OUT, exist_ok=True)

if not os.path.exists('/content/py312/bin/python'):
    !pip -q install uv
    !uv venv /content/py312 --python 3.12 -q
    !uv pip install -q --python /content/py312/bin/python vllm==0.6.6.post1 transformers==4.47.1
!/content/py312/bin/python -c "import vllm, torch; print(vllm.__version__, torch.cuda.is_available())"

%cd {CODE}
!python src/build_corpus.py
!sha256sum corpus/prompts.jsonl
!python tests/test_run_collect.py
!/content/py312/bin/python src/run_collect.py {MODEL} {OUT} --reps 10 --max-hours {MAX_HOURS}

진행 현황 확인. 10/10 되면 MODEL 바꿔서 새 세션에서 수집 셀 실행

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, glob, os
OUT = '/content/drive/MyDrive/C1/results/02_collect'
for d in sorted(glob.glob(f'{OUT}/*/')):
    rows = [json.loads(l) for f in sorted(glob.glob(d + 'chunk_r*.jsonl')) for l in open(f, encoding='utf-8')]
    full = [f for f in glob.glob(d + 'chunk_r*.jsonl') if sum(1 for _ in open(f, encoding='utf-8')) == 200]
    cut = sum(r['finish'] == 'length' for r in rows)
    print(f"{os.path.basename(d[:-1])}: {len(full)}/10 rep, 생성 {len(rows)}, 잘림 {cut}")
print('결과 폴더:', os.listdir(OUT))